# Colab 02 — LLM με RAG (Qwen 2.5 7B, χωρίς εκπαίδευση)

Το LLM αποφασίζει E/S/C/I για κάθε ζεύγος αναζήτησης–προϊόντος. Τρέχουν δύο εκδοχές:
1. **`llm_rag_qwen7b`**: στοιχεία προϊόντος + **4 λυμένα παραδείγματα** που βρίσκει το retrieval από το train
2. **`llm_zeroshot_qwen7b`**: μόνο στοιχεία προϊόντος (για να φανεί πόσο βοηθά το retrieval)

Αξιολόγηση στο σταθερό υποσύνολο **500 αναζητήσεων του test** (~10.000 ζεύγη) και 150 του val (για calibration).

**Πριν ξεκινήσεις:** Runtime → Change runtime type → **T4 GPU**. Μετά **Runtime → Run all**.
Κάθε εκδοχή, όταν τελειώσει, κατεβάζει **αμέσως** το δικό της zip, ώστε να μη χαθεί αν το Colab αποσυνδεθεί αργότερα.
Το smoke test (βήμα 4) τυπώνει ταχύτητα και εκτίμηση χρόνου (ETA).

Τίποτα δεν εγκαθίσταται στον υπολογιστή σου.

## 1. Έλεγχος GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Κώδικας από το GitHub και πακέτα

In [ ]:
%cd /content
!rm -rf repo && git clone --depth 1 https://github.com/CharalamposZisis/Amazon-Recommendation-System.git repo
%cd /content/repo
!pip install -q bitsandbytes accelerate pyyaml

## 3. ESCI (~2GB, στο Colab) και δημιουργία splits
Ίδιο seed (42) με τα τοπικά splits· το κελί ελέγχει ότι βγαίνουν ίδια.

In [ ]:
RAW = "data/esci_data/esci-data/shopping_queries_dataset"
!mkdir -p {RAW}
!wget -q -nc -P {RAW} https://media.githubusercontent.com/media/amazon-science/esci-data/main/shopping_queries_dataset/shopping_queries_dataset_examples.parquet
!wget -q -nc -P {RAW} https://media.githubusercontent.com/media/amazon-science/esci-data/main/shopping_queries_dataset/shopping_queries_dataset_products.parquet
!wget -q -nc -P {RAW} https://raw.githubusercontent.com/amazon-science/esci-data/main/shopping_queries_dataset/shopping_queries_dataset_sources.csv
!python -m src.data --config configs/data.yaml

In [ ]:
import json
info = json.load(open("data/processed/esci_small_us/split_info.json"))["splits"]
expected = {"train": (377719, 18799), "val": (41934, 2089), "test": (None, 8956)}
for s, (rows, queries) in expected.items():
    assert info[s]["queries"] == queries, (s, info[s])
    assert rows is None or info[s]["rows"] == rows, (s, info[s])
print("Τα splits είναι ίδια με τα τοπικά ✔")

## 4. Smoke test (3 αναζητήσεις)
Την πρώτη φορά κατεβαίνει το μοντέλο (~15GB στο Colab, λίγα λεπτά). Δες τις γραμμές `ζεύγη/s · ETA` για να υπολογίσεις τον χρόνο των επόμενων βημάτων.

In [ ]:
!python -m src.llm_rag --config configs/llm_rag_qwen7b.yaml --max-queries 3
!rm -rf results/*_smoke results/*_smoke_calibrated

## 5. LLM + RAG (κύριο πείραμα)

In [ ]:
!python -m src.llm_rag --config configs/llm_rag_qwen7b.yaml
!zip -r -q results_colab02_rag.zip results/llm_rag_qwen7b results/llm_rag_qwen7b_calibrated
from google.colab import files
files.download("results_colab02_rag.zip")

## 6. LLM zero-shot (ablation, χωρίς retrieval)

In [ ]:
!python -m src.llm_rag --config configs/llm_zeroshot_qwen7b.yaml
!zip -r -q results_colab02_zeroshot.zip results/llm_zeroshot_qwen7b results/llm_zeroshot_qwen7b_calibrated
files.download("results_colab02_zeroshot.zip")

## 7. Πίνακας: όλα τα μοντέλα στο ίδιο υποσύνολο
(Τα παλιά μοντέλα δεν έχουν predictions εδώ στο Colab, οπότε ο πλήρης πίνακας βγαίνει τοπικά.)

In [ ]:
!python -m src.report --split test --subset configs/subsets/test_q500.json